# 02. 모델 비교 및 결과 분석

5개 모델 (RF, SVM, MLP, XGBoost, LightGBM)의 성능을 비교하고 리포트용 그래프를 생성합니다.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.model_training import load_data, evaluate_all_models, print_comparison_table
from src.model_training import plot_roc_curves, plot_confusion_matrices
from src.model_training import plot_feature_importance, plot_pr_curves

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

## 1. 데이터 로드 및 모델 평가

In [ ]:
X, y = load_data(feature_set='B')
print(f'Features: {X.shape}')
print(f'Labels: {y.shape}, 독성={y.sum()}, 비독성={(y==0).sum()}')

In [ ]:
# 10-fold Stratified CV 실행 (시간이 걸릴 수 있음)
all_results, roc_data, models = evaluate_all_models(X, y)
best_name = print_comparison_table(all_results)

## 2. ROC Curve 비교

In [ ]:
plot_roc_curves(roc_data, save=True)

## 3. Confusion Matrix

In [ ]:
plot_confusion_matrices(roc_data, save=True)

## 3.5 Precision-Recall Curve

불균형 데이터에서 ROC보다 더 엄격한 평가. DILI처럼 Recall이 중요한 문제에서 특히 유용.

In [ ]:
plot_pr_curves(roc_data, save=True)

## 4. 성능 비교 막대 그래프

In [ ]:
# 모든 지표를 막대 그래프로 비교
metrics = ['auc', 'accuracy', 'f1', 'precision', 'recall']
model_names = list(all_results.keys())

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(model_names))
width = 0.15

colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']

for i, metric in enumerate(metrics):
    means = [all_results[m][metric]['mean'] for m in model_names]
    stds = [all_results[m][metric]['std'] for m in model_names]
    ax.bar(x + i * width, means, width, yerr=stds, label=metric.upper(),
           color=colors[i], alpha=0.8, capsize=3)

ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_title('Model Comparison - All Metrics')
ax.set_xticks(x + width * 2)
ax.set_xticklabels(model_names, rotation=15, ha='right')
ax.legend()
ax.set_ylim(0, 1.1)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/model_comparison_bars.png', dpi=150)
plt.show()

## 4.5 Feature Importance (Random Forest)

어떤 분자 특성이 간독성 예측에 가장 중요한지 확인. 모델 해석력을 보여주는 핵심 시각화.

In [ ]:
# RF 모델을 전체 데이터로 학습하여 Feature Importance 추출
rf_model = models['Random Forest']
rf_model.fit(X, y)

# Feature 이름 로드
feature_names = pd.read_csv('../data/processed/features_B.csv', nrows=0).columns.values
plot_feature_importance(rf_model, feature_names, top_n=20, save=True)

## 5. Feature Set 비교 (선택사항)

In [ ]:
# 다른 feature set으로도 평가해보고 싶다면 아래 코드 실행
# from src.feature_engineering import prepare_features
# prepare_features(feature_set='A')  # Morgan FP only
# prepare_features(feature_set='C')  # Morgan + MACCS + Descriptors
# X_a, y_a = load_data(feature_set='A')
# X_c, y_c = load_data(feature_set='C')
print('Feature set 비교는 시간이 오래 걸릴 수 있습니다.')
print('필요 시 위 코드의 주석을 해제하고 실행하세요.')

## 6. 결과 요약

| 항목 | 내용 |
|------|------|
| 데이터 | TDC DILI (~475 약물) |
| Feature | Morgan FP + Physicochemical descriptors |
| 평가 | 10-fold Stratified CV |
| 최적 모델 | (위 결과 참조) |